<a href="https://colab.research.google.com/github/codealchemist007/flyrank-ml-track/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/codealchemist007/week1_assignment1/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

Unit of analysis: one row = one content item on one calendar day.
The table is fact_content_daily_performance, grain is (report_date, client_hash_id, content_hash_id).

Time window: I'll iterate on March 2026 (month=2026-03) for development and queries.
The full warehouse spans 2025-01-27 to 2026-06-30, but I'll treat June 2026 as a sealed test month.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

FEATURES (knowable at decision time — prior 90 days):
- gsc_impressions_prior90, gsc_clicks_prior90, gsc_avg_position — search visibility
- ga4_sessions_prior90, engagement_rate_prior90, scroll_rate_prior90 — engagement
- content_age_days, word_count, content_type — content metadata
- All from feature window: 2026-01-01 through 2026-03-31 (90 days prior to label)

LABEL / PROXY (future outcome — next 30 days):
- is_declining_label = TRUE if gsc_impressions drop >20% from feature window (Jan-Mar avg)
  to label window (Apr 1-30, 2026)
- Measured on the same content, same client, comparing rolling 30-day means

CONTEXT (for grouping/joining, never as model inputs):
- report_date, client_hash_id, content_hash_id — grain and split keys
- month (partition for iteration on mid-panel months)

EXCLUDED (privacy, product decisions, leakage, incomplete):
- Raw client names, domains, URLs, query text — privacy
- Product flags (health_score, priority_score, action_type) — circular, would leak
- Any search/engagement data from Apr 1-30 used as features — future leakage
- Rows where ga4_data_available = FALSE — GA4 data is zero-filled before client's GA4 start
- Rows where gsc_impressions_prior90 = 0 — no signal to learn from
- Rate columns (ctr, engagement_rate) treated as ×100 percentages, not decimals
- position = 0 treated as missing, not rank zero
- Content types with >80% missingness in keyword data — will use has_keyword_flag instead

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

I ran three verification queries against the real warehouse data (March 2026,
month=2026-03) to prove the contract claims above, not just assume them.

Query 1 - Grain check: total_rows and unique (report_date, client, content)
combinations both came out to 9,841,378 - an exact match. This confirms the
grain really is one row per content item per day, with no duplicates.

Query 2 - Row count + date span: 9,841,378 rows, spanning 2026-03-01 to
2026-03-31 - confirms I'm looking at the correct month partition, fully covered.

Query 3 - Availability (using IS TRUE): all 9,841,378 rows have GSC impressions
data (100%), but only 413,966 rows (about 4.2%) have ga4_data_available = TRUE.
This is a real and important finding - GA4/engagement signals are far sparser
than search signals in this slice, which limits how much I can rely on
engagement-based features for most pages.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

from google.colab import userdata
import duckdb

hf_token = userdata.get('HF_TOKEN')
con = duckdb.connect()
con.sql(f"CREATE SECRET hf_token (TYPE HUGGINGFACE, TOKEN '{hf_token}');")

# Query 1
grain_check = con.sql("""
    SELECT COUNT(*) as total_rows,
           COUNT(DISTINCT (report_date, client_hash_id, content_hash_id)) as unique_combos
    FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')
""").to_df()
print(grain_check)

# Query 2
row_count_dates = con.sql("""
    SELECT COUNT(*) as row_count,
           MIN(report_date) as earliest_date,
           MAX(report_date) as latest_date
    FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')
""").to_df()
print(row_count_dates)

# Query 3
availability = con.sql("""
    SELECT
        COUNT(*) as total_rows,
        SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) as ga4_available_rows,
        SUM(CASE WHEN gsc_impressions IS NOT NULL THEN 1 ELSE 0 END) as impressions_available
    FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')
""").to_df()
print(availability)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   total_rows  unique_combos
0     9841378        9841378
   row_count earliest_date latest_date
0    9841378    2026-03-01  2026-03-31


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   total_rows  ga4_available_rows  impressions_available
0     9841378            413966.0              9841378.0


Five features, built from the March 2026 slice. Each one is knowable at the
decision moment (before we'd know the future outcome), so none of these leak
the label.

In [4]:
features = con.sql("""
    SELECT
        content_hash_id,
        client_hash_id,
        report_date,
        gsc_impressions,
        gsc_clicks,
        gsc_avg_position,
        CASE WHEN gsc_impressions > 0
             THEN gsc_clicks * 1.0 / gsc_impressions
             ELSE NULL END as ctr,
        ga4_data_available
    FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')
    LIMIT 1000
""").to_df()

print(features.head(10))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

            content_hash_id           client_hash_id report_date  \
0  content_b7e512995f79d5a6  client_73cda7b4e4f265ea  2026-03-01   
1  content_05597932fe4da067  client_73cda7b4e4f265ea  2026-03-01   
2  content_7a105f548d9c6916  client_73cda7b4e4f265ea  2026-03-01   
3  content_905aa32a0230694e  client_73cda7b4e4f265ea  2026-03-01   
4  content_a3ea9792f793ec72  client_73cda7b4e4f265ea  2026-03-01   
5  content_36c36abc7650d7af  client_73cda7b4e4f265ea  2026-03-01   
6  content_a7da352b73b02668  client_73cda7b4e4f265ea  2026-03-01   
7  content_05434271b257bb68  client_73cda7b4e4f265ea  2026-03-01   
8  content_d056587ff7faca0c  client_73cda7b4e4f265ea  2026-03-01   
9  content_bfd1e41c2af250c8  client_73cda7b4e4f265ea  2026-03-01   

   gsc_impressions  gsc_clicks  gsc_avg_position       ctr  ga4_data_available  
0               20           0          3.350000  0.000000                <NA>  
1                1           0          0.000000  0.000000                <NA>  
2       

The trap: I'll deliberately add a label-derived column as a "feature" and
watch a quick score jump toward perfect - then remove it and show the
honest number. This proves why trend_direction can never be a real feature.

In [5]:
# Build a REAL label: did each page's impressions decline from early March to late March?
monthly_data = con.sql("""
    SELECT
        content_hash_id,
        SUM(CASE WHEN report_date <= '2026-03-15' THEN gsc_impressions ELSE 0 END) as impressions_early,
        SUM(CASE WHEN report_date > '2026-03-15' THEN gsc_impressions ELSE 0 END) as impressions_late,
        SUM(CASE WHEN report_date <= '2026-03-15' THEN gsc_clicks ELSE 0 END) as clicks_early,
        AVG(CASE WHEN report_date <= '2026-03-15' THEN gsc_avg_position END) as avg_position_early
    FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')
    GROUP BY content_hash_id
    HAVING impressions_early > 0
""").to_df()

# Real, observed label: did impressions decline from early to late March?
monthly_data["is_declining"] = (monthly_data["impressions_late"] < monthly_data["impressions_early"]).astype(int)

print(monthly_data["is_declining"].value_counts())
print(monthly_data.shape)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

is_declining
0    85395
1    66586
Name: count, dtype: int64
(151981, 6)


In [6]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

y = monthly_data["is_declining"]

# LEAKED VERSION: sneaks in impressions_late, which is literally what the label was built from
X_leaked = monthly_data[["impressions_early", "clicks_early", "avg_position_early", "impressions_late"]].fillna(0)

X_train, X_test, y_train, y_test = train_test_split(X_leaked, y, test_size=0.3, random_state=42)
model_leaked = LogisticRegression(max_iter=1000).fit(X_train, y_train)
preds_leaked = model_leaked.predict_proba(X_test)[:, 1]
score_leaked = roc_auc_score(y_test, preds_leaked)

print(f"LEAKED score (impressions_late sneaked in as a feature): {score_leaked:.4f}")

LEAKED score (impressions_late sneaked in as a feature): 1.0000


In [7]:
# HONEST VERSION: only use data knowable BEFORE the label window (early March)
X_honest = monthly_data[["impressions_early", "clicks_early", "avg_position_early"]].fillna(0)

X_train, X_test, y_train, y_test = train_test_split(X_honest, y, test_size=0.3, random_state=42)
model_honest = LogisticRegression(max_iter=1000).fit(X_train, y_train)
preds_honest = model_honest.predict_proba(X_test)[:, 1]
score_honest = roc_auc_score(y_test, preds_honest)

print(f"HONEST score (impressions_late removed): {score_honest:.4f}")
print(f"\nDifference: {score_leaked - score_honest:.4f} - this gap is the leakage")

HONEST score (impressions_late removed): 0.5847

Difference: 0.4153 - this gap is the leakage


As expected, sneaking impressions_late into the features produced a perfect
score of 1.0000 - but this is fake, not a real pattern. impressions_late is
literally the exact data the is_declining label was computed from, so the
model wasn't learning anything - it was just basically reading the answer back to itself.

Removing it and using only early-March data (impressions_early, clicks_early,
avg_position_early) - things I'd genuinely know before the label window -
dropped the score to 0.5847, barely above the 0.5 random-guessing baseline.

The 0.4153 gap between these two scores is the leakage, made visible with
real numbers. This proves hands-on why trend_direction/trend_pct (and by
extension, any future-window data) can never be used as features - a leaked
feature makes a model look perfect on paper while being completely useless
in a real prediction scenario, where you'd never have "the future" available
at the decision time.

The honest 0.5847 score also tells me something real: three basic early-window
signals alone aren't strongly predictive of decline on their own - which
makes sense for a first pass, and points toward needing richer features
(engagement signals, trend patterns, content metadata) in later weeks.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

The clearest limitation of this data slice is GA4 sparsity - only about 4.2% of rows in March 2026 have ga4_data_available = TRUE, meaning engagement signals (sessions, engagement_rate, scroll_rate) are only usable for a small fraction of pages. Any feature built from GA4 data will have heavy missingness that isn't random - it's tied to whether a client had GA4 tracking set up at all during that period, not whether the page had "zero engagement."
Beyond that, this snapshot only covers one month (March 2026) out of a much longer history (Jan 2025 - June 2026), and client history depth varies a lot.
Also some clients have over a year of data, others much less. So any pattern I find in this one month might not generalize across the full panel, and I'd need to check other months before trusting a signal too heavily. This slice also excludes the most recent 3 days of any month by design, and the final month (June 2026) is deliberately kept sealed as a test set, so I can't peek at it while developing features or labels now.   

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.